## Transformación y limpieza de datos

### Objetivo
Este notebook aplica las transformaciones necesarias sobre los datos extraídos para prepararlos para su carga en la base de datos analítica.

Las operaciones incluyen:
- **Tipado de columnas**: conversión de IDs numéricos a cadena (`str`) para tratarlos como variables categóricas.
- **Mapeo de territorios**: asignación del `id_territorio` desde la tabla dimensión a los DataFrames de hechos (constituidas y disueltas), y normalización de nombres (minúsculas, guiones bajos).
- **Normalización de texto**: estandarización de nombres de sectores, meses, razones de disolución y tipos de medida.
- **Eliminación de duplicados**: filtrado de filas "Mercantiles" que agregan información ya presente en los desgloses por tipo societario.
- **Abreviaturas**: conversión de tipos societarios a siglas (S.A., S.L., S.Com./S.C.).

### Metodología
1. **Carga** de los CSV desde `../files/data_raw/`.
2. **Transformaciones** aplicadas mediante funciones del módulo `src.transformacion` y mapeos manuales.
3. **Exportación** de los datasets procesados a `../files/data_processed/` para su consumo en la fase de carga.

In [ ]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación del módulo de transformacion
from src.transformacion import trans_str
from src.transformacion import trans_normal
# Configuración para visualizar todas las columnas
pd.set_option('display.max_columns', None)

Abrimos los ficheros

In [ ]:
df_empr_const = pd.read_csv('../files/data_raw/empresas_constituidas.csv')
df_empr_dis = pd.read_csv('../files/data_raw/empresas_disueltas.csv')
df_ipc = pd.read_csv('../files/data_raw/ipc.csv')
df_sectores_ipc = pd.read_csv('../files/data_raw/sectores_ipc.csv')
df_territorio = pd.read_csv('../files/data_raw/territorio.csv')
df_tiempo = pd.read_csv('../files/data_raw/tiempo.csv')
df_tipo_medida = pd.read_csv('../files/data_raw/tipo_medida.csv')

Transformamos id's, año y mes a string ya que son falsas numericas (actuan como categoricas)

In [ ]:
# Listas de columnas a transformar de cada DataFrame
lista_const = ['id_const', 'id_tiempo']
lista_dis = ['id_dis', 'id_tiempo']
lista_ipc = ['id_tiempo', 'id_territorio', 'id_sector', 'id_medida']
lista_sector_ipc = ['id_sector']
lista_territorio = ['id_territorio']
lista_tiempo = ['id_tiempo', 'anio', 'mes']
lista_tipo_medida = ['id_medida']

In [ ]:
trans_str.int_a_str(df_empr_const, lista_const)
trans_str.int_a_str(df_empr_dis, lista_dis)
trans_str.int_a_str(df_ipc, lista_ipc)
trans_str.int_a_str(df_sectores_ipc, lista_sector_ipc)
trans_str.int_a_str(df_territorio, lista_territorio)
trans_str.int_a_str(df_tiempo, lista_tiempo)
trans_str.int_a_str(df_tipo_medida, lista_tipo_medida)

In [ ]:
df_empr_const.info()

In [ ]:
df_empr_dis.info()

In [ ]:
df_ipc.info()

In [ ]:
df_sectores_ipc.info()

In [ ]:
df_territorio.info()

In [ ]:
df_territorio.head()

In [ ]:
df_tiempo.info()

In [ ]:
df_tipo_medida.info()

- Cambiar los Nombres de las comunidades autónomas, sin acentos, con minúsculas y separación con guión bajo ('_')
- DataFrames: empresas_constituidas, empresas_disueltas y territorio

In [ ]:
# Creamos el diccionario de mapeo con tus especificaciones exactas
mapeo_territorios_nombre = {
    "Nacional": "nacional",
    "Andalucía": "andalucia",
    "Aragón": "aragon",
    "Asturias, Principado de": "principado_de_asturias",
    "Balears, Illes": "islas_baleares",
    "Canarias": "canarias",
    "Cantabria": "cantabria",
    "Castilla y León": "castilla_y_leon",
    "Castilla - La Mancha": "castilla_la_mancha",
    "Cataluña": "cataluna",
    "Comunitat Valenciana": "comunidad_valenciana",
    "Extremadura": "extremadura",
    "Galicia": "galicia",
    "Madrid, Comunidad de": "comunidad_de_madrid",
    "Murcia, Región de": "region_de_murcia",
    "Navarra, Comunidad Foral de": "comunidad_foral_de_navarra",
    "País Vasco": "pais_vasco",
    "Rioja, La": "la_rioja",
    "Ceuta": "ceuta",
    "Melilla": "melilla"
}

In [ ]:
# Diccionario territorio -> id_territorio, usando el propio df_territorio
mapeo_territorios = dict(zip(df_territorio['nombre_territorio'], df_territorio['id_territorio']))

# Aplicar el mapeo
df_empr_const['id_territorio'] = df_empr_const['territorio'].map(mapeo_territorios)
df_empr_dis['id_territorio'] = df_empr_dis['territorio'].map(mapeo_territorios)

In [ ]:
df_empr_const.sample(10)

In [ ]:
df_empr_dis.sample(10)

In [ ]:
df_empr_dis.drop(columns=["territorio"],inplace=True)

In [ ]:
df_empr_const.drop(columns=["territorio"],inplace=True)

In [ ]:
#df_empr_const['territorio'] = df_empr_const['territorio'].replace(mapeo_territorios)
#df_empr_dis['territorio'] = df_empr_dis['territorio'].replace(mapeo_territorios)
df_territorio['nombre_territorio'] = df_territorio['nombre_territorio'].replace(mapeo_territorios_nombre)

In [ ]:
df_territorio['nombre_territorio'].unique()

Normalizamos resto de columnas (minúsculas, separación con guión bajo ('_'))

In [ ]:
# Así se aplica una función a los DATOS de una columna
df_empr_dis["razon"] = df_empr_dis["razon"].apply(trans_normal.normalizar_col)
df_sectores_ipc["nombre_sector"] = df_sectores_ipc["nombre_sector"].apply(trans_normal.normalizar_col)
df_tiempo["nombre_mes"] = df_tiempo["nombre_mes"].apply(trans_normal.normalizar_col)
df_tipo_medida["nombre_medida"] = df_tipo_medida["nombre_medida"].apply(trans_normal.normalizar_col)

In [ ]:
df_empr_dis.sample(10)

In [ ]:
df_sectores_ipc.sample(10)

In [ ]:
df_tiempo.sample(10)

In [ ]:
df_tipo_medida.sample(4)

Eliminar en empresas_contituidas las tipo mercantiles, son sumatorias del resto de tipo y nos duplican los datos

In [ ]:
# Eliminamos las filas que contienen "Mercantiles"
df_empr_const = df_empr_const[df_empr_const['tipo'] != 'Mercantiles']

In [ ]:
df_empr_const['tipo'].unique()

In [ ]:
df_empr_const.shape

Cambiamos el nombre de las sociedades por sus acrónimos

In [ ]:
# Creamos el diccionario de mapeo con tus especificaciones exactas
dicc_siglas = {
    'Sociedades de responsabilidad limitada': 'S.L.',
    'Sociedades anónimas': 'S.A.',
    'S. Comanditarias y S. Colectivas': 'S.Com./S.C.'
}

In [ ]:
# Aplicamos el cambio a la columna 'tipo'
df_empr_const['tipo'] = df_empr_const['tipo'].replace(dicc_siglas)

In [ ]:
df_empr_const['tipo'].unique()

Guardamos los csv's procesados

In [ ]:
df_empr_const.to_csv('../files/data_processed/empresas_constituidas.csv', index=False)
df_empr_dis.to_csv('../files/data_processed/empresas_disueltas.csv', index=False)
df_ipc.to_csv('../files/data_processed/ipc.csv', index=False)
df_sectores_ipc.to_csv('../files/data_processed/sectores_ipc.csv', index=False)
df_territorio.to_csv('../files/data_processed/territorio.csv', index=False)
df_tiempo.to_csv('../files/data_processed/tiempo.csv', index=False)
df_tipo_medida.to_csv('../files/data_processed/tipo_medida.csv', index=False)